# 03 — Baseline Model (Week 4)

**Model:** Linear Regression on `log(ClosePrice)`, trained on the X = 18 months before the test month
(X chosen in Week 3 on validation months only) and evaluated **once** on the most recent complete month.

Preprocessing is the leakage-safe pipeline from Week 3, plus one fix found here: numeric features are clipped
to their 0.5th/99.5th percentiles **learned on the training set** (`QuantileClipper`). Raw MLS fields contain
entry errors (a 24.9-billion sq ft lot, 500 garage spaces, −143 parking spaces) that a linear model
extrapolates into absurd prices.

Logic lives in `src/idx_avm/baseline.py`; `scripts/run_baseline.py` runs the same steps end to end.

In [ ]:
import sys, json
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from idx_avm import config
from idx_avm.pipeline import prepare_dataset
from idx_avm.baseline import run_baseline, breakdowns, residual_summary

X_MONTHS = json.loads((config.PROCESSED_DIR / "split_metadata.json").read_text())["chosen_window_months"]
data = prepare_dataset()
TEST_MONTH = data.months[-1]
print(f"{len(data.df):,} cleaned rows | test month {TEST_MONTH} | training window {X_MONTHS} months")

## 1. Train and evaluate on the test month

In [ ]:
result = run_baseline(data.df, TEST_MONTH, X_MONTHS)
print("Train months:", [str(m) for m in result.train_months])
pd.Series(result.metrics).round(3)

## 2. Effect of the train-only feature clipping

Same model, same split, with and without clipping. MdAPE barely moves (the typical home was fine) but R² and RMSE improve a lot, because a handful of absurd predictions disappear.

In [ ]:
raw = run_baseline(data.df, TEST_MONTH, X_MONTHS, clip_features=False)
pd.DataFrame({"raw features": raw.metrics, "clipped (baseline)": result.metrics}).T.round(3)

## 3. Rolling-origin backtest

Re-run the identical pipeline with the two previous months as test months. Stable metrics mean the baseline isn't tuned to one lucky month.

In [ ]:
rows = []
for k in (2, 1, 0):
    r = result if k == 0 else run_baseline(data.df, TEST_MONTH - k, X_MONTHS)
    rows.append({"test_month": str(TEST_MONTH - k), **r.metrics})
pd.DataFrame(rows).round(3)

## 4. Where the error is

Per-band R² is not meaningful (each band has a narrow price range), so read MdAPE/MAPE here.

In [ ]:
parts = breakdowns(result.test)
display(parts["by_price_band"][["price_range", "MdAPE", "MAPE", "MAE", "n"]].round(2))
display(parts["by_county"][["MdAPE", "MAPE", "MAE", "n"]].round(2).head(15))
pd.Series(residual_summary(result.test)).round(1)

In [ ]:
t = result.test
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(t[config.TARGET], t["Predicted"], s=3, alpha=0.3)
lim = [t[config.TARGET].min(), t[config.TARGET].max()]
axes[0].plot(lim, lim, "k--", lw=1)
axes[0].set(xscale="log", yscale="log", xlabel="Actual ClosePrice", ylabel="Predicted", title="Predicted vs actual (test month)")
pct = 100 * (t["Predicted"] - t[config.TARGET]) / t[config.TARGET]
axes[1].hist(pct.clip(-60, 60), bins=80)
axes[1].axvline(0, color="k", lw=1)
axes[1].set(xlabel="% error (clipped at ±60)", title="Percentage error distribution")
plt.tight_layout(); plt.show()

## 5. Baseline record

| Metric (test month 2026-04, n = 12,337) | Value |
|---|---|
| MdAPE (headline) | 11.11% |
| MAPE | 15.57% |
| R² | 0.811 |
| MAE | $212,780 |
| Within 10% of sale price | 45.7% |

Before the clipping fix: R² 0.685, MdAPE 11.24%. Backtest MdAPE at 2026-02 / 2026-03 / 2026-04: 11.12% / 11.11% / 11.11%.

**Strengths:** stable across months; unbiased overall (median signed error ≈ 0.1%); best on $575K–$1.1M homes (MdAPE ≈ 9–10%).

**Weaknesses:** worst on the top price quintile (MdAPE 16.4%), where a straight line in log-space can't capture
how location and size interact; it can't learn non-linear effects without hand-built features.
Week 5 tree models (Decision Tree, Random Forest) are the next step.